## Build Drivers Dimension

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01-Environment_Config

In [0]:
%run ../00-common/04-Gold_Helpers

In [0]:
drivers_table = f"{catalog_name}.{silver_schema}.drivers"
nationality_table = f"{catalog_name}.{gold_schema}.ref_nationality_region"

gold_table = f"{catalog_name}.{gold_schema}.dim_drivers"

In [0]:
from pyspark.sql import functions as f

In [0]:
drivers_df = spark.read.table(drivers_table).filter((f.col("batch_id") == v_batch_id))

nationality_region_df = spark.read.table(nationality_table)

In [0]:
joined_df = drivers_df.join(
    nationality_region_df, drivers_df.nationality == nationality_region_df.nationality, how="left"
).select(
    drivers_df.driver_id, 
    drivers_df.driver_name,
    drivers_df.date_of_birth, 
    drivers_df.nationality, 
    nationality_region_df.Region.alias("nationality_region")
    )

In [0]:
columns = [
    "driver_name",
    "date_of_birth",
    "nationality",
    "nationality_region"
]

write_to_gold(joined_df, gold_table, "trg.driver_id = src.driver_id", columns)